# 05 Verifikasi Formulasi CARE KM


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('05_Verifikasi_Formulasi_CARE_KM')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)


# Synthetic categorical data: deliberately small and independent of the research dataset.
X_toy = np.array([
    ['A','L','x'], ['A','L','y'], ['A','M','x'], ['A','M','y'],
    ['B','M','x'], ['B','M','z'], ['B','H','z'], ['B','H','y'],
    ['C','L','x'], ['C','L','z'], ['C','H','y'], ['C','H','z'],
], dtype=object)
K_TOY = 3

# SMD definition: delta=0 if equal, 1 otherwise; object distance is sum over attributes.
def smd_manual(a, b):
    return int(sum(x != y for x, y in zip(a, b)))

D_manual = np.array([[smd_manual(x, z) for z in X_toy[:K_TOY]] for x in X_toy])
D_lib = ck.smd_matrix(X_toy, X_toy[:K_TOY])
assert np.array_equal(D_manual, D_lib)
print('SMD direct verification: PASS')

cao_modes = ck.cao_init(X_toy, K_TOY)
cao_cost = ck.initial_cost(X_toy, cao_modes)
care_modes, trace, path = ck.care_init(
    X_toy, K_TOY, return_trace=True, return_path=True, **CARE_CONFIG
)
care_cost = ck.initial_cost(X_toy, care_modes)

trace_df = pd.DataFrame(trace)
derived_budget = ck.derive_refinement_budget(CARE_CONFIG['candidate_budget'])

assert derived_budget == int(np.ceil(np.log2(CARE_CONFIG['candidate_budget'])))
assert np.array_equal(path[0], cao_modes)
assert len(trace_df) <= derived_budget
if len(trace_df):
    assert (trace_df['gain'] > 0).all()
    assert (trace_df['cost_after'] < trace_df['cost_before']).all()
assert care_cost <= cao_cost + 1e-12

# Encoded representation must preserve SMD exactly.
X_enc = ck._encode_categorical_for_care(X_toy)
D_enc = ck._smd_matrix_encoded(X_enc, X_enc[:K_TOY])
assert np.array_equal(D_lib, D_enc)

stopping_reason = (
    'NO_POSITIVE_GAIN'
    if len(trace_df) < derived_budget
    else 'DERIVED_REFINEMENT_BUDGET_REACHED'
)

verification = pd.DataFrame([{
    'toy_rows': len(X_toy),
    'toy_features': X_toy.shape[1],
    'k': K_TOY,
    'parent_initializer': 'Cao',
    'candidate_budget': CARE_CONFIG['candidate_budget'],
    'derived_refinement_budget': derived_budget,
    'cao_initial_cost': cao_cost,
    'care_initial_cost': care_cost,
    'accepted_exchanges': len(trace_df),
    'stopping_reason': stopping_reason,
    'smd_direct_equals_library': True,
    'encoded_smd_equals_object_smd': True,
    'path_starts_from_cao': True,
    'care_nonworsening_vs_cao': bool(care_cost <= cao_cost + 1e-12),
}])

el.save_csv(verification, artifact_path('02_formulation_verification.csv','05_Verifikasi_Formulasi_CARE_KM'))
el.save_csv(trace_df, artifact_path('02_exchange_trace.csv','05_Verifikasi_Formulasi_CARE_KM'))
el.save_csv(pd.DataFrame(care_modes), artifact_path('02_toy_care_initial_modes.csv','05_Verifikasi_Formulasi_CARE_KM'))
display(verification)
display(trace_df)
print('STATUS: PASS')
